# Eigenmode simulation of two qubits with resonators and a coupler

In [ ]:
# Iteratively optimize the two qubit chip with the following divisions:
%load_ext autoreload
%autoreload 2
from copy import deepcopy
import names as n
import design as d
from qdesignoptimizer.utils.chip_generation import create_chip_base

import mini_studies as ms
import optimization_targets as ot

import parameter_targets as pt
import plot_settings as ps

from qdesignoptimizer.design_analysis import (
    DesignAnalysis,
    DesignAnalysisState,
    merge_partitioned_simulation,
)
from qdesignoptimizer.utils.utils import get_save_path
from qdesignoptimizer.utils.names_parameters import param, param_nonlin
from qdesignoptimizer.design_analysis import save_optimization_results
from qdesignoptimizer.design_analysis_types import SimulationResults
from qdesignoptimizer.sim_plot_progress import plot_progress

from qdesignoptimizer.utils.utils import close_ansys

import qdesignoptimizer

design, gui = create_chip_base(
    n.CHIP_NAME,
    d.chip_type,
    open_gui=True,
    design_variables_file="design_variables.json",
)
d.render_qiskit_metal_design(design, gui)


RES_QB_CPLR = "res_qb_cplr"
QB_CPLR_QB = "qb_cplr_qb"
CPLR_QB_RES = "cplr_qb_res"

nbr_iterations = 11
group_passes = 8
delta_f = 0.001

run_counter = 0
optimization_results = []
minimization_results = []

PLOT_SETTINGS = ps.PLOT_SETTINGS_TWO_QB
RENDER_QISKIT_METAL = lambda design: d.render_qiskit_metal_design(design, gui)

groups = [n.NBR_1, n.NBR_2]

for i in range(nbr_iterations):
    # Use a clean state from previous iteration to demonstrate that all subsystems can be optimized in parallel
    prev_iteration_variables = (
        deepcopy(optimization_results[-1]["design_variables"])
        if optimization_results
        else {}
    )
    prev_iteration_system_optimized_params = (
        deepcopy(optimization_results[-1]["system_optimized_params"])
        if optimization_results
        else {}
    )
    if prev_iteration_system_optimized_params is None:
        prev_iteration_system_optimized_params = {}
    start_new_result_entry = True
    for partition in [RES_QB_CPLR, CPLR_QB_RES, QB_CPLR_QB]:

        state_partition = DesignAnalysisState(
            design,
            RENDER_QISKIT_METAL,
            pt.PARAM_TARGETS,
            system_optimized_params=deepcopy(prev_iteration_system_optimized_params),
        )
        MINI_STUDY = ms.get_mini_study_2qb_resonator_coupler_partitioned(partition)

        if partition == RES_QB_CPLR:
            include_param_in_update = [
                param(n.QUBIT_1, n.FREQ),
                param(n.RESONATOR_1, n.FREQ),
                param_nonlin(n.QUBIT_1, n.QUBIT_1),
                param_nonlin(n.QUBIT_1, n.RESONATOR_1),
            ]
        elif partition == CPLR_QB_RES:
            include_param_in_update = [
                param(n.QUBIT_2, n.FREQ),
                param(n.RESONATOR_2, n.FREQ),
                param_nonlin(n.QUBIT_2, n.QUBIT_2),
                param_nonlin(n.QUBIT_2, n.RESONATOR_2),
            ]
        elif partition == QB_CPLR_QB:
            include_param_in_update = [
                param(n.COUPLER_12, n.FREQ),
                param_nonlin(n.COUPLER_12, n.COUPLER_12),
                param_nonlin(n.COUPLER_12, n.QUBIT_1),
                param_nonlin(n.COUPLER_12, n.QUBIT_2),
            ]

        # Include all targets which will be used in all partitions such that
        # the update of the design variables know which parameters will change
        opt_targets = ot.get_opt_targets_2qubits_resonator_coupler(
            groups=groups,
            opt_target_qubit_freq=True,
            opt_target_qubit_anharm=True,
            opt_target_resonator_freq=True,
            opt_target_resonator_qubit_chi=True,
            opt_target_coupler_freq=True,
            opt_target_coupler_anharm=True,
            opt_target_coupler_qubit_chi=True,
        )

        design_analysis = DesignAnalysis(
            state_partition,
            mini_study=MINI_STUDY,
            opt_targets=opt_targets,
            save_path=None,
            update_design_variables=False,
            plot_settings=None,  # Plot after merging iteration results
            is_part_of_partitioned_optimization=True,
        )

        design_analysis.update_nbr_passes(group_passes)
        design_analysis.update_delta_f(delta_f)

        # Use a clean state from previous iteration to demonstrate that all subsystems can be optimized in parallel
        design_analysis.optimize_target(
            deepcopy(prev_iteration_variables),
            deepcopy(prev_iteration_system_optimized_params),
        )
        # design_analysis.screenshot(gui=gui, run=run_counter)

        minimization_results.extend(design_analysis.minimization_results)
        state_partition.system_optimized_params = (
            design_analysis.system_optimized_params
        )

        merge_partitioned_simulation(
            optimization_results_to_update=optimization_results,
            state_to_fetch_from=state_partition,
            optimization_results_to_fetch_from=design_analysis.optimization_results,
            opt_targets=opt_targets,
            include_param_in_update=include_param_in_update,
            start_new_result_entry=start_new_result_entry,
        )
        save_optimization_results(
            simulation_result=SimulationResults(
                optimization_results=optimization_results,
                system_target_params=pt.PARAM_TARGETS,
                plot_settings=PLOT_SETTINGS,
                design_analysis_version=qdesignoptimizer.__version__,
            ),
            updated_design_vars=design_analysis.design.variables,
            save_path=get_save_path("out/", n.CHIP_NAME),
        )

        plot_progress(
            opt_results=[optimization_results],
            system_target_params=pt.PARAM_TARGETS,
            plot_settings=PLOT_SETTINGS,
        )
        start_new_result_entry = False

        run_counter += 1

## Results

### Eigenmodes

In [ ]:
design_analysis.get_eigenmode_results()

### Cross Kerr

In [ ]:
design_analysis.get_cross_kerr_matrix(iteration=-1)

## Update parameters

In [ ]:
design_analysis.overwrite_parameters()